In [8]:
import pandas as pd
import sqlite3

In [9]:
#create SQLite database and connection. Create cursor

conn = sqlite3.connect('Survivor.db')
cursor = conn.cursor()

In [10]:
# read in CSVs as data frames

df_castaways_clean = pd.read_csv("castaways_clean.csv")
df_advantage_details_clean = pd.read_csv("advantage_details_clean.csv")
df_advantage_movement_clean = pd.read_csv("advantage_movement_clean.csv")

In [11]:
# Create tables

cursor.execute("PRAGMA foreign_keys = ON")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Castaways (
    Version TEXT,
    Version_Season TEXT,
    Season INTEGER,
    Full_Name TEXT,
    Castaway_Id TEXT,
   "Order" INTEGER,
    Result TEXT,
    Jury_Status TEXT,
    Place INTEGER,
    Jury BOOL,
    Finalist BOOL,
    Winner BOOL,
    Season_Castaway_Id TEXT PRIMARY KEY,
    Finale_Categorization TEXT,
    Era TEXT
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Advantage_Details (
    Version TEXT,
    Version_Season TEXT,
    Season INTEGER,
    Advantage_Id TEXT,
    Advantage_Type TEXT,
    Clue_Details TEXT,
    Location_Found TEXT,
    Conditions TEXT,
    Season_Advantage_Id TEXT PRIMARY KEY          
);
""")

cursor.execute ("""
CREATE TABLE IF NOT EXISTS Advantage_Movement (             
    Version TEXT,
    Version_Season TEXT,
    Season INTEGER,
    Castaway TEXT,
    Castaway_Id TEXT,
    Advantage_Id TEXT,
    Sequence_Id TEXT,
    Event TEXT,
    Success TEXT,
    Season_Advantage_Id TEXT,
    Season_Castaway_Id TEXT,
    Movement_Id TEXT PRIMARY KEY,    
    FOREIGN KEY (Season_Advantage_Id) REFERENCES Advantage_Details(Season_Advantage_Id),
    FOREIGN KEY (Season_Castaway_Id) REFERENCES Castaways(Season_Castaway_Id)
);
""")


In [12]:
#add data to SQLite tables

df_castaways_clean.to_sql("Castaways", conn, if_exists="replace", index=False)
df_advantage_details_clean.to_sql("Advantage_Details", conn, if_exists="replace",index=False)
df_advantage_movement_clean.to_sql("Advantage_Movement", conn, if_exists="replace",index=False)


557

In [15]:
#Verify data was inserted

print(pd.read_sql_query("SELECT * FROM Castaways", conn))
print(pd.read_sql_query("SELECT * FROM Advantage_Details", conn))
print(pd.read_sql_query("SELECT * FROM Advantage_Movement", conn))

    Version Version_Season  Season          Full_Name Castaway_Id  Order  \
0        US           US01       1  Sonja Christopher      US0001    1.0   
1        US           US01       1      B.B. Andersen      US0002    2.0   
2        US           US01       1    Stacey Stillman      US0003    3.0   
3        US           US01       1        Ramona Gray      US0004    4.0   
4        US           US01       1          Dirk Been      US0005    5.0   
..      ...            ...     ...                ...         ...    ...   
912      US           US49      49              TBD14      US0747    NaN   
913      US           US49      49              TBD15      US0748    NaN   
914      US           US49      49              TBD16      US0749    NaN   
915      US           US49      49              TBD17      US0750    NaN   
916      US           US49      49              TBD18      US0751    NaN   

            Result Jury_Status  Place  Jury  Finalist  Winner  \
0    1st voted out    

In [17]:
#Query function

def query(query:str):
    return pd.read_sql(query, conn)

In [20]:
#Test query function

all_castaways = """SELECT * FROM Castaways"""
df_all_castaways = query(all_castaways)
df_all_castaways

,Version,Version_Season,Season,Full_Name,Castaway_Id,Order,Result,Jury_Status,Place,Jury,Finalist,Winner,Season_Castaway_Id,Finale_Categorization,Era
0,US,US01,1,Sonja Christopher,US0001,1.0,1st voted out,None,16,0,0,0,US01US0001,Voted out pre-jury,Old School (pre-advantage)
1,US,US01,1,B.B. Andersen,US0002,2.0,2nd voted out,None,15,0,0,0,US01US0002,Voted out pre-jury,Old School (pre-advantage)
2,US,US01,1,Stacey Stillman,US0003,3.0,3rd voted out,None,14,0,0,0,US01US0003,Voted out pre-jury,Old School (pre-advantage)
3,US,US01,1,Ramona Gray,US0004,4.0,4th voted out,None,13,0,0,0,US01US0004,Voted out pre-jury,Old School (pre-advantage)
4,US,US01,1,Dirk Been,US0005,5.0,5th voted out,None,12,0,0,0,US01US0005,Voted out pre-jury,Old School (pre-advantage)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
912,US,US49,49,TBD14,US0747,NaN,None,None,5,0,1,1,US49US0747,Winner,New Era
913,US,US49,49,TBD15,US0748,NaN,None,None,4,0,1,1,US49US0748,Winner,New Era
914,US,US49,49,TBD16,US0749,NaN,None,None,3,0,1,1,US49US0749,Winner,New Era
915,US,US49,49,TBD17,US0750,NaN,None,None,2,0,1,1,US49US0750,Winner,New Era


In [38]:
#Query: Percentage of Winners who had an advantage
Winner_Percentage = """
SELECT
    CASE
      WHEN Advantage_Movement.Season_Castaway_Id IS NOT NULL THEN 'Advantage'
      ELSE 'No Advantage'
    END AS advantage_status,
COUNT(DISTINCT Castaways.Castaway_Id) AS count
FROM Castaways
LEFT JOIN Advantage_Movement
    ON Castaways.Season_Castaway_Id = Advantage_Movement.Season_Castaway_Id
WHERE Castaways.finale_categorization = 'Winner' AND Castaways.season<49
GROUP BY advantage_status;
"""

query(Winner_Percentage)

,advantage_status,count
0,Advantage,21
1,No Advantage,26


In [ ]:
#Visualization of the percentage of winners who had an advantage

In [ ]:
#Query: Advantages and Place
Advantage_Place_Correlation = """
SELECT
FROM

"""

In [ ]:
# Visualization of correlation between number of advantages and place

In [ ]:
#Query

Finale_Categorization_Advantage = """
SELECT Castaways.Finale_Categorization,
COUNT(DISTINCT Castaways.Castaway_Id) AS count
FROM Castaways
INNER JOIN Advantage_Movement
    ON Castaways.Season_Castaway_Id = Advantage_Movement.Season_Castaway_Id
WHERE Castaways.season<49
GROUP BY Castaways.Finale_Categorization;
"""

query(Finale_Categorization_Advantage)

,Finale_Categorization,count
0,Finalist,32
1,Jury,112
2,Voted out pre-jury,28
3,Winner,21


In [ ]:
# visualization of number of individuals with an advantage grouped by finale categorization

In [50]:
#Query

Finale_Categorization_Played_Advantage = """
SELECT Castaways.Finale_Categorization,
COUNT(DISTINCT Castaways.Castaway_Id) AS count
FROM Castaways
INNER JOIN Advantage_Movement
    ON Castaways.Season_Castaway_Id = Advantage_Movement.Season_Castaway_Id
WHERE Castaways.season<49 AND Advantage_Movement.Event = 'Played'
GROUP BY Castaways.Finale_Categorization;
"""

query(Finale_Categorization_Played_Advantage)

,Finale_Categorization,count
0,Finalist,27
1,Jury,61
2,Voted out pre-jury,6
3,Winner,17


In [ ]:
# visualization of number of individuals with an advantage played grouped by finale categorization

In [51]:
#Query

Finale_Categorization_Successfully_Played_Advantage = """
SELECT Castaways.Finale_Categorization,
COUNT(DISTINCT Castaways.Castaway_Id) AS count
FROM Castaways
INNER JOIN Advantage_Movement
    ON Castaways.Season_Castaway_Id = Advantage_Movement.Season_Castaway_Id
WHERE Castaways.season<49 AND Advantage_Movement.Event = 'Played'AND Advantage_Movement.Success = 'Yes'
GROUP BY Castaways.Finale_Categorization;
"""

query(Finale_Categorization_Successfully_Played_Advantage)

,Finale_Categorization,count
0,Finalist,6
1,Jury,28
2,Voted out pre-jury,3
3,Winner,10


In [ ]:
#visualization of number of individuals with an advantage successfully played grouped by finale categorization

In [ ]:
#visualization with era incorporated

In [ ]:
#close the connection

connection.close()